# S2 · от сенсорного лога к матрице признаков

Измерения IMU и GNSS → общая временная сетка → окна → матрица признаков.

## Исходный лог и дефекты времени

Диагностика времени IMU подсчитывает нечисловые метки (`non_finite`), повторы (`duplicate`) и обратные переходы (`backward`). Один `NaN` увеличивает `non_finite` на единицу.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

# Jupyter обычно открывает kernel в notebooks; запуск из starter также поддержан.
STARTER_ROOT = next(
    p for p in (Path.cwd(), Path.cwd().parent)
    if (p / "src" / "ml_sau" / "sensor.py").is_file()
)
sys.path.insert(0, str(STARTER_ROOT / "src"))
from ml_sau.sensor_source import G0, generate_sensor_log
from ml_sau.sensor import (
    FEATURE_NAMES, GAP_TOLERANCE_S, WindowBatch,
    assemble_quality_report, linear_interpolation, write_sensor_artifacts,
)

raise NotImplementedError("S2 block 1: диагностика времени")

assert timestamp_report(np.array([0.0, 0.1, 0.1]))["duplicate"] == 1
assert timestamp_report(np.array([0.0, 0.2, 0.1]))["backward"] == 1
assert timestamp_report(np.array([np.nan]))["non_finite"] == 1
assert timestamp_report(np.array([])) == {"non_finite": 0, "duplicate": 0, "backward": 0}
assert time_quality["non_finite"] == time_quality["backward"] == 0


## Единицы и общие часы

Два канала акселерометра задают удельную силу по осям x и z датчика, третий — угловую скорость. GNSS даёт скорость. Параметры часов заданы формулой `t_gnss = 1.00022 * t_common + 0.18`. Пересчёт единиц и обращение этой формулы приводят каналы к общим единицам и времени.

In [ ]:
raise NotImplementedError("S2 block 2: единицы и общие часы")

np.testing.assert_allclose(180.0 * np.pi / 180.0, np.pi)
np.testing.assert_allclose(gnss_t[[0, -1]], [0.0, 30.0])
assert np.all(np.diff(imu_t) > 0)


## Интерполяция и разрывы

`linear_interpolation` проверяет формы массивов и выполняет `np.interp` по каналам; за пределами записи возвращает `NaN`. Промежуточные точки внутри слишком длинных разрывов также помечаются `NaN`. Допуск `1e-9` с учитывает округление чисел: интервалы GNSS номинально равны 0.10 с. Для низкочастотных сигналов этого набора используется общая сетка 50 Гц.

In [ ]:
raise NotImplementedError("S2 block 3: маска длинных разрывов")

# Один и тот же числовой пример: значения концов сохраняются при обоих пределах.
toy_t = np.array([0.0, 2.0])
toy_v = np.array([0.0, 4.0])
toy_grid = np.array([-1.0, 0.0, 1.0, 2.0, 3.0])
np.testing.assert_allclose(interpolate_with_gap_mask(toy_t, toy_v, toy_grid, 2.0), [np.nan, 0.0, 2.0, 4.0, np.nan])
np.testing.assert_allclose(interpolate_with_gap_mask(toy_t, toy_v, toy_grid, 0.5), [np.nan, 0.0, np.nan, 4.0, np.nan])
assert np.isfinite(speed_aligned).all()


## Окна и их происхождение

Окна содержат по 100 отсчётов с шагом 50. Для каждого окна сохраняется время начала. На сетке 50 Гц длительность окна по числу отсчётов — 2 с; первая и последняя временные метки внутри него различаются на 1.98 с.

In [ ]:
raise NotImplementedError("S2 block 4: окна и время их начала")

np.testing.assert_allclose(batch.start_s[~batch.valid], [16.0, 17.0, 18.0])
assert batch.values.shape == (29, 100, 4)


## Признаки и матрица X

Каждое допустимое окно описывается четырьмя агрегатами. RMS включает постоянную составляющую, стандартное отклонение вычисляется относительно среднего. При `ddof=0` сумма квадратов отклонений делится на число отсчётов окна. Строки матрицы `X` содержат признаки окон; целевые метки для обучения берутся из отдельного источника.

In [ ]:
raise NotImplementedError("S2 block 5: вычисление признаков")

assert X.shape == (int(batch.valid.sum()), len(FEATURE_NAMES))
assert np.isfinite(X).all()


## Сравнение правил и сохранение результата

Сравниваются пределы разрыва 0.10 и 0.50 с: число допустимых окон, их положение и значения признаков. Результат для предела 0.10 с сохраняется в `s2-quality.json`, `s2-features.csv` и `s2-windows.csv`. Значения внутри разрывов рассчитаны интерполяцией.

In [ ]:
raise NotImplementedError("S2 block 6: сравнение правил обработки")

# Служебная запись уже подготовлена: вычисления сделаны в предыдущих блоках.
report = assemble_quality_report(log, time_quality, grid_s, batch, X, max_gap_s)
write_sensor_artifacts(STARTER_ROOT / "reports", report, X, batch)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("saved:", STARTER_ROOT / "reports" / "s2-features.csv")
